# Sparse Koopman Operators and Worst-Case Reconstruction

This notebook uses two **power-user** training terms from `koopman_graph.losses`:

- `KoopmanSparsityLoss` — element-wise $L_1$ (or smoothed $L_p$, $p<1$) on the
  latent operator. Ordinary models penalize `koopman.matrix`; networked
  `koopman="graph"` models penalize **`K_self` and `K_nbr` parameters only**.
- `WorstCaseReconstructionLoss` — max-over-nodes MSE (batch $L_\infty$-style).
  This is a **robust training term**, not a PAC / generalization certificate.

## Interpretability caveats (read first)

- Sparse latent $K$ is **not** a sparse physical adjacency matrix.
- For graph operators, parameter sparsity on $K_{\mathrm{self}}/K_{\mathrm{nbr}}$ is
  **not** sparsity of the topology-bound effective operator
  $I\otimes K_{\mathrm{self}} + \hat{A}\otimes K_{\mathrm{nbr}}$.
- SINDy-style sparse regression (Brunton, Proctor & Kutz, 2016) identifies
  sparse ODEs in chosen observables; here the penalty is a soft regularizer
  on learned Koopman factors, not an automatic governing-equation discovery
  pipeline in node coordinates.

Enable the terms with `LossWeights(sparsity=..., worst_case=...)`. They are
**not** on the root `koopman_graph` façade.

In [ ]:
import warnings

from tqdm.std import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

import os

import matplotlib

if os.environ.get("PYTEST_CURRENT_TEST"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import torch

try:
    from IPython import get_ipython

    if get_ipython() is not None and not os.environ.get("PYTEST_CURRENT_TEST"):
        get_ipython().run_line_magic("matplotlib", "inline")
except (ImportError, NameError):
    pass

from koopman_graph import GNNDecoder, GNNEncoder, GraphKoopmanModel
from koopman_graph.datasets import SyntheticDynamicGraphBenchmark
from koopman_graph.losses import KoopmanSparsityLoss
from koopman_graph.training import LossWeights

torch.manual_seed(42)
sequence = SyntheticDynamicGraphBenchmark.generate(
    num_nodes=8,
    num_timesteps=16 if os.environ.get("PYTEST_CURRENT_TEST") else 24,
    in_channels=3,
    topology="path",
    seed=42,
)
print(sequence)


## Sparsity weight sweep

Train identical architectures from the same seed with increasing
`LossWeights.sparsity`. We report the fraction of $|K_{ij}| < 10^{-2}$ after
fitting — higher weight should yield a sparser latent operator.

In [ ]:
def sparsity_fraction(matrix: torch.Tensor, threshold: float = 1e-2) -> float:
    return float((matrix.detach().abs() < threshold).float().mean().item())


weights = [0.0, 0.5, 2.0]
fractions = []
matrices = []

for sparsity_w in weights:
    torch.manual_seed(0)
    model = GraphKoopmanModel(
        encoder=GNNEncoder(in_channels=3, hidden_channels=16, latent_dim=8),
        decoder=GNNDecoder(latent_dim=8, hidden_channels=16, out_channels=3),
        latent_dim=8,
        time_step=0.1,
    )
    history = model.fit(
        sequence,
        epochs=12 if os.environ.get("PYTEST_CURRENT_TEST") else 40,
        lr=5e-2,
        loss_weights=LossWeights(reconstruction=1.0, sparsity=sparsity_w),
    )
    k = model.koopman.matrix.detach().cpu()
    frac = sparsity_fraction(k)
    fractions.append(frac)
    matrices.append(k)
    print(
        f"sparsity={sparsity_w:g}: frac(|K|<1e-2)={frac:.3f}, "
        f"final L_sp={history.sparsity_loss[-1]:.4f}"
    )

assert fractions[0] < fractions[-1], "higher sparsity weight should sparsify K"

fig, axes = plt.subplots(1, len(weights), figsize=(9, 2.8), constrained_layout=True)
for ax, w, k, frac in zip(axes, weights, matrices, fractions):
    im = ax.imshow(k.abs().numpy(), cmap="magma", vmin=0.0)
    ax.set_title(f"w={w:g}\nfrac={frac:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="|K_ij|")
plt.show()


## Optional worst-case term

`LossWeights.worst_case` adds the max-over-nodes reconstruction penalty.
Use it when hard nodes dominate error; do **not** treat the resulting fit as
having a certified generalization or $L_\infty$ bound.

In [ ]:
torch.manual_seed(1)
model_wc = GraphKoopmanModel(
    encoder=GNNEncoder(in_channels=3, hidden_channels=16, latent_dim=8),
    decoder=GNNDecoder(latent_dim=8, hidden_channels=16, out_channels=3),
    latent_dim=8,
    time_step=0.1,
)
history_wc = model_wc.fit(
    sequence,
    epochs=8 if os.environ.get("PYTEST_CURRENT_TEST") else 20,
    lr=5e-2,
    loss_weights=LossWeights(reconstruction=1.0, sparsity=0.25, worst_case=0.1),
)
print(
    "final losses:",
    f"recon={history_wc.reconstruction_loss[-1]:.4f},",
    f"sparsity={history_wc.sparsity_loss[-1]:.4f},",
    f"worst_case={history_wc.worst_case_loss[-1]:.4f}",
)
assert history_wc.worst_case_loss[-1] >= 0.0

# Standalone factor check (ordinary dense K).
sp = KoopmanSparsityLoss()(model_wc.koopman)
print(f"standalone KoopmanSparsityLoss={float(sp):.4f}")


## Takeaways

1. Raise `LossWeights.sparsity` to encourage sparse latent operators.
2. For `koopman="graph"`, inspect `K_self` / `K_nbr` — not `effective_matrix` —
   when interpreting the sparsity target.
3. `worst_case` is optional robust training; keep claims empirical.
4. Import sparsity / worst-case helpers from `koopman_graph.losses`.